In [1]:
import os
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    spherical_grid, spherical_radial_sampling
import equiv_dens.utils.base as utils
from equiv_dens.training.model_loader import load_model

import numpy as np
from functools import partial
import argparse
import copy
import ase.io
%load_ext autoreload
%autoreload 2

Use "numpy" for Fourier Transform


/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


In [2]:
args, hyperparam_args = parse_command_line_arguments(arg_file='water_dummy.txt')
print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified
directory = args.restart  # load directory name
# load latest checkpoint

# create directories
# write command line arguments to file (useful for reproducibility)
checkpoint = None
latest_checkpoint = 0
step = 0
restore = False
data_split_indices = None

# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = args.use_gpu and torch.cuda.is_available()

# load dataset(s)
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")
args.use_gpu = False
if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose)
torch.manual_seed(0)
np.random.seed(0)
args.restart = None

model = load_model(args, dataset)

type dtype <class 'torch.dtype'>
args np dir datasets/h2o_dynamic_centered.npy
args use gpu True
loading density fromdatasets/h2o_dynamic_pyscf_dft.npy...
loading atoms fromdatasets/h2o_dynamic_centered.npy...
Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'energy', 'forces', 'full_hamiltonian', 'overlap_matrix', 'core_hamiltonian', 'atom_types', 'atom_numbers'])
grid fn functools.partial(<function spherical_grid at 0x7f5f635dae18>, level=2)
len atom types 3
atom numbers 8
level 2
finished init
cg_matrix shape torch.Size([121, 121, 121])
args energy_unit_in kcal
args energy_unit_out kcal
conversions in <function kcal_to_kcal at 0x7f5f635f0400>
conversions out <function kcal_to_kcal at 0x7f5f635f0400>
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
creating embedding
init_coeffs None
orbital basis {8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8,

In [11]:
sample = dataset.get_properties(0)
res = model(sample)

sph norms [tensor([[1., 1., 1., 1., 1., 1.]]), tensor([[3.0000, 3.0000, 3.0000, 3.0000, 3.0000, 3.0000]]), tensor([[5.0000, 5.0000, 5.0000, 5.0000, 5.0000, 5.0000]]), tensor([[7.0000, 7.0000, 7.0000, 7.0000, 7.0000, 7.0000]]), tensor([[9.0000, 9.0000, 9.0000, 9.0000, 9.0000, 9.0000]]), tensor([[11.0000, 11.0000, 11.0000, 11.0000, 11.0000, 11.0000]])]
vs norms [tensor([[1.9445e-06, 1.8250e-06, 9.6904e-07, 1.6740e-06, 1.0853e-06, 1.6740e-06]],
       grad_fn=<MeanBackward1>), tensor([[8.9966e-07, 1.1496e-06, 1.1238e-06, 1.9677e-06, 8.1492e-07, 1.9677e-06]],
       grad_fn=<MeanBackward1>)]
a norms [tensor([[0.0078, 0.0078, 0.0078, 0.0078, 0.0078, 0.0078]],
       grad_fn=<MeanBackward1>), tensor([[0.0234, 0.0234, 0.0234, 0.0234, 0.0234, 0.0234]],
       grad_fn=<MeanBackward1>)]
rbf norms [tensor([0.0004, 0.0004, 0.0004, 0.0004, 0.0004, 0.0004],
       grad_fn=<MeanBackward1>)]
rbf tensor([[[[1.3825e-29, 1.2069e-27, 5.2268e-26, 1.4971e-24, 3.1902e-23,
           5.3949e-22, 7.5407e-21, 8